# 06. Kaggle Training — CarDD damage segmentation with YOLO11n

Этот notebook рассчитан на Kaggle Notebook с GPU и уже готовым YOLO-датасетом CarDD, добавленным через `Add input`.

Что делает:
- использует путь к уже готовому YOLO-датасету, который ты явно задашь после подключения input
- читает существующий `data.yaml`
- обучает `yolo11n-seg.pt` на тех же 6 damage-классах
- использует упрощённый recipe, близкий к сильному внешнему baseline, но реалистичный для ограничений Kaggle GPU

**Источник данных**
- Используй уже готовый YOLO dataset CarDD
- В ячейке ниже просто укажи `DATASET_ROOT` вручную, например `/kaggle/input/datasets/gabrielfcarvalho/cardd-with-yolo-annotations-images-labels`

**Упрощённые гиперпараметры**
- модель: `yolo11n-seg.pt`
- `imgsz=896`
- `epochs=80`
- `batch=8`
- `optimizer=AdamW`
- `cls=0.3`, `dfl=1.7`
- без слишком тяжёлых трюков, чтобы notebook стабильно запускался на одном Kaggle GPU

In [ ]:
!pip install -q ultralytics pyyaml wandb


In [ ]:
import json
import os
import shutil
from pathlib import Path

import yaml
import wandb
from ultralytics import YOLO


In [ ]:
WORKDIR = Path('/kaggle/working')
OUTPUT_ROOT = WORKDIR / 'cardd_yolo11n_seg'
DATASET_ROOT = Path('/kaggle/input/datasets/gabrielfcarvalho/cardd-with-yolo-annotations-images-labels')  # поменяй при необходимости
SOURCE_DATA_YAML = DATASET_ROOT / 'data.yaml'
RESOLVED_DATA_YAML = OUTPUT_ROOT / 'data_resolved.yaml'
RUNS_DIR = OUTPUT_ROOT / 'runs'
REPORTS_DIR = OUTPUT_ROOT / 'reports'
WEIGHTS_DIR = OUTPUT_ROOT / 'weights'
for p in [RUNS_DIR, REPORTS_DIR, WEIGHTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RUN_NAME = 'yolo11n_cardd_seg_kaggle'
WANDB_PROJECT = 'cardd-damage-seg'
WANDB_ENTITY = None  # при необходимости впиши entity/team
WANDB_SECRET_NAME = 'WB'

CFG = {
    'model': 'yolo11n-seg.pt',
    'epochs': 80,
    'imgsz': 896,
    'batch': 8,
    'patience': 20,
    'optimizer': 'AdamW',
    'lr0': 5e-4,
    'lrf': 0.01,
    'warmup_epochs': 5,
    'dropout': 0.05,
    'cls': 0.3,
    'dfl': 1.7,
    'mosaic': 0.5,
    'mixup': 0.05,
    'copy_paste': 0.0,
    'hsv_h': 0.015,
    'hsv_s': 0.5,
    'hsv_v': 0.25,
    'degrees': 5.0,
    'translate': 0.05,
    'scale': 0.2,
    'fliplr': 0.5,
    'multi_scale': True,
    'amp': True,
    'overlap_mask': True,
    'workers': 4,
    'seed': 42,
}

print(CFG)


In [ ]:
wandb_run = None
wandb_key = None

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret(WANDB_SECRET_NAME)
    print(f'Loaded W&B key from Kaggle secret: {WANDB_SECRET_NAME}')
except Exception as exc:
    print(f'Kaggle secret {WANDB_SECRET_NAME} not available: {exc}')

USE_WANDB = bool(wandb_key)

if USE_WANDB:
    wandb.login(key=wandb_key)
    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=RUN_NAME,
        config={
            **CFG,
            'dataset_root': str(DATASET_ROOT),
            'source_data_yaml': str(SOURCE_DATA_YAML),
            'resolved_data_yaml': str(RESOLVED_DATA_YAML),
        },
        job_type='train',
    )
    print('W&B run:', wandb_run.name)
else:
    print('W&B disabled. Add Kaggle secret WB if tracking is needed.')


In [ ]:
assert DATASET_ROOT.exists(), f'Missing dataset root: {DATASET_ROOT}'
assert SOURCE_DATA_YAML.exists(), f'Missing data.yaml: {SOURCE_DATA_YAML}'

for split in ['train', 'val', 'test']:
    assert (DATASET_ROOT / split).exists(), f'Missing split dir: {DATASET_ROOT / split}'

print('DATASET_ROOT =', DATASET_ROOT)
print('SOURCE_DATA_YAML =', SOURCE_DATA_YAML)
print(SOURCE_DATA_YAML.read_text(encoding='utf-8'))

In [ ]:
data_yaml = yaml.safe_load(SOURCE_DATA_YAML.read_text(encoding='utf-8'))
print(json.dumps(data_yaml, indent=2, ensure_ascii=False))

if isinstance(data_yaml.get('names'), dict):
    class_names = [data_yaml['names'][k] for k in sorted(data_yaml['names'])]
else:
    class_names = list(data_yaml.get('names', []))

split_dirs = {
    split: {
        'images': DATASET_ROOT / split / 'images',
        'labels': DATASET_ROOT / split / 'labels',
    }
    for split in ['train', 'val', 'test']
}

for split, paths in split_dirs.items():
    assert paths['images'].exists(), f"Missing images dir for {split}: {paths['images']}"
    assert paths['labels'].exists(), f"Missing labels dir for {split}: {paths['labels']}"

resolved_data_yaml = dict(data_yaml)
resolved_data_yaml['path'] = str(DATASET_ROOT)
for split in ['train', 'val', 'test']:
    resolved_data_yaml[split] = str(split_dirs[split]['images'])

RESOLVED_DATA_YAML.write_text(yaml.safe_dump(resolved_data_yaml, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('RESOLVED_DATA_YAML =', RESOLVED_DATA_YAML)
print(RESOLVED_DATA_YAML.read_text(encoding='utf-8'))
print('class_names =', class_names)

In [ ]:
def count_files(path: Path, patterns):
    total = 0
    for pattern in patterns:
        total += len(list(path.glob(pattern)))
    return total

stats = {}
for split in ['train', 'val', 'test']:
    image_dir = split_dirs[split]['images']
    label_dir = split_dirs[split]['labels']
    image_count = count_files(image_dir, ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'])
    label_count = len(list(label_dir.glob('*.txt')))
    stats[split] = {'images': image_count, 'labels': label_count}

report = {
    'dataset_root': str(DATASET_ROOT),
    'source_data_yaml': str(SOURCE_DATA_YAML),
    'resolved_data_yaml': str(RESOLVED_DATA_YAML),
    'class_names': class_names,
    'split_dirs': {k: {kk: str(vv) for kk, vv in val.items()} for k, val in split_dirs.items()},
    'stats': stats,
}
(REPORTS_DIR / 'dataset_stats.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
report

In [ ]:
model = YOLO(CFG['model'])
results = model.train(
    data=str(RESOLVED_DATA_YAML),
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    imgsz=CFG['imgsz'],
    epochs=CFG['epochs'],
    batch=CFG['batch'],
    patience=CFG['patience'],
    optimizer=CFG['optimizer'],
    lr0=CFG['lr0'],
    lrf=CFG['lrf'],
    warmup_epochs=CFG['warmup_epochs'],
    dropout=CFG['dropout'],
    cls=CFG['cls'],
    dfl=CFG['dfl'],
    mosaic=CFG['mosaic'],
    mixup=CFG['mixup'],
    copy_paste=CFG['copy_paste'],
    hsv_h=CFG['hsv_h'],
    hsv_s=CFG['hsv_s'],
    hsv_v=CFG['hsv_v'],
    degrees=CFG['degrees'],
    translate=CFG['translate'],
    scale=CFG['scale'],
    fliplr=CFG['fliplr'],
    multi_scale=CFG['multi_scale'],
    overlap_mask=CFG['overlap_mask'],
    amp=CFG['amp'],
    workers=CFG['workers'],
    seed=CFG['seed'],
    pretrained=True,
    val=True,
    plots=True,
    device=0,
)
print(results.save_dir)


In [ ]:
run_dir = Path(results.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'

best_out = WEIGHTS_DIR / 'best_yolo11n_cardd_seg_kaggle.pt'
last_out = WEIGHTS_DIR / 'last_yolo11n_cardd_seg_kaggle.pt'
metadata_out = WEIGHTS_DIR / 'metadata_yolo11n_cardd_seg_kaggle.json'
report_out = REPORTS_DIR / 'dataset_stats.json'

shutil.copy2(best_pt, best_out)
if last_pt.exists():
    shutil.copy2(last_pt, last_out)

metadata = {
    'model': CFG['model'],
    'dataset_source': str(DATASET_ROOT),
    'classes': class_names,
    'weights': str(best_out),
    'train_cfg': CFG,
    'source_run_dir': str(run_dir),
}
metadata_out.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

if USE_WANDB and wandb_run is not None:
    artifact = wandb.Artifact('yolo11n-cardd-seg-weights', type='model')
    artifact.add_file(str(best_out), name='best_yolo11n_cardd_seg_kaggle.pt')
    if last_out.exists():
        artifact.add_file(str(last_out), name='last_yolo11n_cardd_seg_kaggle.pt')
    artifact.add_file(str(metadata_out), name='metadata_yolo11n_cardd_seg_kaggle.json')
    if report_out.exists():
        artifact.add_file(str(report_out), name='dataset_stats.json')
    wandb.log_artifact(artifact)

print('best_pt =', best_out)
print('metadata =', metadata_out)


In [ ]:
best_model = YOLO(str(WEIGHTS_DIR / 'best_yolo11n_cardd_seg_kaggle.pt'))
val_metrics = best_model.val(data=str(RESOLVED_DATA_YAML), split='val', imgsz=CFG['imgsz'], batch=max(1, CFG['batch'] // 2), device=0)
print(val_metrics.results_dict)

if USE_WANDB and wandb_run is not None and hasattr(val_metrics, 'results_dict'):
    safe_metrics = {k.replace('/', '_'): float(v) for k, v in val_metrics.results_dict.items() if isinstance(v, (int, float))}
    wandb.log(safe_metrics)
    wandb.finish()
